# OJCS-2026-09-0983: Cloud Evaluation Suite
## Benchmarking Unquantized FP16 (3–4B) and Larger (7B/8B) Models on Free GPU Tiers (Colab / Kaggle T4)

This self-contained notebook runs the primary LLM preference elicitation diagnostic across unquantized FP16 checkpoints and larger model classes:
- `Qwen/Qwen2.5-3B-Instruct` (FP16)
- `meta-llama/Llama-3.2-3B-Instruct` (FP16)
- `Qwen/Qwen2.5-7B-Instruct` (FP16 / 8-bit)

It tests whether the behavioral dissociation (abstention vs directive commitment) and position/payoff sensitivities persist under unquantized weights and larger scale.

In [ ]:
# Step 1: Environment Check
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Free VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Step 2: Install Dependencies
!pip install -q transformers accelerate bitsandbytes pandas scipy statsmodels

In [ ]:
# Step 3: Clone Repository or Download Stimuli
!git clone https://github.com/CodeShrek/LLM-Preference-Diagnostics.git ./repo
%cd repo
!ls -la experiments/m3v2/m3v2_stimuli.json

### Benchmark 1: Unquantized FP16 Qwen 2.5 3B

In [ ]:
!python experiments/cloud_eval/run_cloud_benchmarks.py \
    --model_id "Qwen/Qwen2.5-3B-Instruct" \
    --torch_dtype "float16" \
    --num_dilemmas 5 \
    --output_csv "results/cloud_qwen3b_fp16.csv"

### Benchmark 2: Unquantized FP16 Llama 3.2 3B
*(Requires Hugging Face token for Llama-3.2 gating)*

In [ ]:
import getpass
hf_token = getpass.getpass("Enter HuggingFace Token (optional, for Llama-3.2): ")

!python experiments/cloud_eval/run_cloud_benchmarks.py \
    --model_id "meta-llama/Llama-3.2-3B-Instruct" \
    --torch_dtype "float16" \
    --num_dilemmas 5 \
    --hf_token "$hf_token" \
    --output_csv "results/cloud_llama3b_fp16.csv"

### Benchmark 3: Scaled 7B Model (Qwen 2.5 7B Instruct)

In [ ]:
!python experiments/cloud_eval/run_cloud_benchmarks.py \
    --model_id "Qwen/Qwen2.5-7B-Instruct" \
    --torch_dtype "float16" \
    --num_dilemmas 5 \
    --output_csv "results/cloud_qwen7b_fp16.csv"

### Step 4: Cross-Model Diagnostic Comparison

In [ ]:
import pandas as pd
import glob

files = glob.glob("results/cloud_*.csv")
summary = []
for f in sorted(files):
    df = pd.read_csv(f)
    tot = len(df)
    indiff = (df['parsed_response'] == 'Indifferent').sum()
    np = df[df['ratio'] != '1:1']
    committed_np = np[np['parsed_response'].isin(['Option A', 'Option B'])]
    dom = committed_np['dominant_chosen'].sum() if len(committed_np) > 0 else 0
    summary.append({
        "File": f,
        "Total": tot,
        "Abstention (%)": f"{indiff/tot*100:.1f}%",
        "Committed NP": len(committed_np),
        "Dominant Accuracy (%)": f"{dom/len(committed_np)*100:.1f}%" if len(committed_np) > 0 else "N/A"
    })

print(pd.DataFrame(summary).to_string(index=False))